# 강의 05 · 실습 3 — 운영 장치 · (5) 고난도 II


## 1. 문제상황

- 구름월드 안내 서비스의 하루 예산은 0.002달러로 정해졌습니다. 0.002달러는 실습에서 초과를 바로 보기 위해 작게 잡은 값입니다. 오늘 들어온 질문은 8개이고, 그중 2개는 길고 비교를 요구하는 질문입니다.
- 운영팀은 예산을 넘긴 뒤에도 손님을 그냥 돌려보낼 수 없습니다. 예산이 남아 있으면 모델이 답하고, 예산이 다 떨어지면 「잠시 뒤 다시 문의해 주세요」라는 정해진 안내문으로 대체하려 합니다.
- 비용을 아끼려면 짧은 질문은 경량 모델로 보내고, 같은 안내문 접두는 캐시가 붙게 해야 합니다.
- 운영팀은 질문 8개가 어떤 모델로 처리됐는지, 몇 번째부터 안내문으로 대체됐는지, 하루 총비용이 얼마인지를 한 표로 보고 싶어 합니다.


## 2. 문제와 목표

- **문제**: 예산·티어링·캐싱이 따로따로 있고, 예산이 떨어졌을 때의 대체 동작이 없습니다. 질문 8개의 처리 결과를 한 표로 볼 수 없습니다.
- **목표**
  - 예산 장치·티어링·캐싱을 하나의 처리 함수로 묶어 질문 8개를 차례로 처리합니다.
    - 예산: 하루 0.002달러. 예산이 떨어진 뒤의 답은 안내문 「잠시 뒤 다시 문의해 주세요.」
    - 티어링: 질문이 30자보다 길거나 「비교」「설명」「추천」이 들어 있으면 고성능, 아니면 경량 모델
    - 캐싱: 모든 호출의 시스템 메시지는 FAQ를 열두 번 반복해 붙인 긴 안내문(맨 앞에 실행 시각 표시). 메시지 목록은 시스템 메시지와 사용자 메시지 두 개로 직접 만듭니다
    - 질문 8개(짧은 6개, 긴 2개)는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다
  - 질문마다 고른 모델·비용·누적 비용·캐시 히트·상태를 표로 출력하고, 마지막에 총비용과 대체된 질문 수를 출력합니다.
    - 상태 열: 모델이 답했으면 「모델 답」, 예산이 떨어져 안내문으로 대체했으면 「안내문 대체」
- **목표 달성 여부의 판정 기준**:
  - 표에 질문 8개가 한 줄씩 출력되고,
  - 예산이 남아 있는 동안은 짧은 질문이 경량 모델, 긴 비교 질문이 고성능 모델로 골라지며,
  - 누적 비용이 0.002달러를 넘긴 뒤의 질문은 상태가 「안내문 대체」로 출력되고 비용이 0입니다.
  - 같은 모델의 두 번째 호출부터 캐시 히트가 0보다 큽니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex03_s5_diagram.svg)


## 4. 단계별 요구사항

(이 단에서는 요구사항을 학생이 직접 씁니다. 「2. 문제와 목표」의 목표와 「3. 워크플로우 다이어그램」만 보고, 단계마다 무엇을 만들어야 하는지 번호 목록으로 적은 뒤 「6. 코드 — 스텝바이스텝」의 코드를 작성합니다.)


## 5. 코드 골격

(이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다. 쓰지 않는 단은 이유를 적습니다.)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델 이름 세 개를 정합니다. 이 실습의 모델 호출은 `litellm.completion`을 직접 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- 모델 세 개는 기본 모델(`PRIMARY`), 경량 모델(`CHEAP`), 고성능 모델(`HIGH`)입니다. 모델 이름은 공급자 이름을 앞에 붙인 문자열 그대로 쓰고, 별칭이나 중계 서버는 쓰지 않습니다.
- `litellm.suppress_debug_info = True`와 `logging` 설정 한 줄은 오류가 났을 때 litellm이 화면에 출력하는 안내 배너와 오류 로그를 끕니다. 동작에는 영향이 없습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import logging
import os
import time   # time — time.strftime("%H%M%S")로 긴 안내문 맨 앞의 실행 시각 표시를 만듭니다
import warnings

from dotenv import load_dotenv, find_dotenv

import litellm
from langsmith import Client, traceable   # Client — Client().flush()로 남은 런을 LangSmith로 보냅니다
from langsmith.run_helpers import get_current_run_tree

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")   # 추적 라이브러리가 내는 직렬화 경고를 화면에서 감춥니다
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex03"

PRIMARY = "openai/gpt-5.6-luna"
CHEAP = "openai/gpt-4o-mini"
HIGH = "openai/gpt-5.6-terra"
print("모델 세 개:", PRIMARY, CHEAP, HIGH)

# 주어진 자료: 처리할 질문 목록 QUESTIONS — 값을 그대로 씁니다
QUESTIONS = [
    "운영 시간이 어떻게 되나요?", "야간개장은 언제 하나요?",
    "평일 오후에 아이 둘과 가는데 자유이용권과 야간개장 중 어느 쪽이 유리한지 비교해서 설명해 주세요.",
    "자유이용권 환불이 되나요?", "주차는 몇 시간 무료인가요?",
    "야간개장 날 퍼레이드 시간과 운영 시간을 함께 설명해 주세요. 주차 요금도 알려 주세요.",
    "안녕하세요!", "환불 규정 알려 주세요",
]


운영 장치를 붙일 안내 서비스입니다. 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_messages`가 질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만들고, 모델 호출 함수 `litellm.completion`은 `traceable`로 감싸 llm 런에 기록되어 있습니다(답한 모델 이름을 런 메타데이터에 적습니다). 서비스가 도는 것을 먼저 확인합니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

GUIDE = ("너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
         "인사말에는 짧은 인사로 답한다. FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.")


def make_messages(question: str, guide: str = GUIDE) -> list:
    """질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만든다."""
    return [{"role": "system", "content": guide + "\n=== FAQ ===\n" + FAQ},
            {"role": "user", "content": question}]


_completion = litellm.completion


@traceable(run_type="llm", name="litellm.completion", metadata={"ls_provider": "openai"})
def completion(**kwargs):
    """이 실습에서 쓰는 관측 계측. 답한 모델 이름을 런 메타데이터에 적는다."""
    res = _completion(**kwargs)
    get_current_run_tree().metadata["ls_model_name"] = res.model
    return res


litellm.completion = completion

Q = "자유이용권 환불이 되나요?"
res = litellm.completion(model=PRIMARY, messages=make_messages(Q))
print(res.model, "→", res.choices[0].message.content[:60])

In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 표에 질문 8개가 한 줄씩 출력됩니다.
2. 짧은 질문은 경량 모델, 길고 비교·설명을 요구하는 질문은 고성능 모델로 골라집니다.
3. 누적 비용이 예산을 넘긴 뒤의 질문은 상태가 「안내문 대체」이고 비용이 0입니다.
4. 같은 모델의 두 번째 호출부터 캐시 히트가 0보다 큽니다. 캐시는 모델마다 따로 붙으므로 고성능 모델의 첫 호출은 캐시 히트가 0입니다. 마지막 줄에 총비용과 대체된 질문 수가 출력됩니다.

네 가지가 모두 확인되면 완성입니다. 대체된 질문 수를 보고 「예산을 얼마로 잡아야 하루를 넘길 수 있는가」를 다음 질문으로 이어 갑니다.
